# Experiment 6: Pause after the heads decide where to look, change where they look, resume

Each head of the look-around part works in two steps:
1. **Decide where to look:** it compares the word's query with the keys of the earlier words, giving **attention weights**
   (one share per earlier word, adding up to 100%).
2. **Collect:** it mixes the values of the earlier words using those shares, giving its 64-number head output.

In Experiment 5 we paused **after step 2** and changed what a head collected.
Here we pause **between step 1 and step 2**: the heads have decided where to look, but have not collected anything yet.
We then change the attention weights directly, e.g. *"head 8.11, don't look at ' Paris'"* or *"look only at ' Paris'"*, and resume.

In this notebook we:
1. Check that splitting the heads into "decide" and "collect" still gives exactly the normal answer.
2. **PAUSE** after the attention weights are computed, and **save** them together with the values.
3. **MODIFY** the attention weights of the two heads found in Experiment 5 (8.11 and 11.2): block, force, or redirect them.
4. **RESUME** by collecting with the changed weights and running the rest of the model, **without rerunning** anything earlier.
5. Cut the link between ' speak' and ' Paris' in one block at a time, and in all blocks.
6. Cut **every** later word's link to ' Paris', so the model can never read it.

- **Part A:** head 8.11, how much it looks at ' Paris'
- **Part B:** head 11.2, and both heads together
- **Part C:** cutting the link to ' Paris' block by block

## Step 0: Install (Colab only)
Tested with PyTorch 2.14.1 and Transformers 5.20.2. Uncomment only if something breaks.

In [ ]:
# !pip install -q transformers==5.20.2

## Step 1: Load GPT-2 small

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_grad_enabled(False)  # prediction only, no training
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
gpt2 = model.transformer
NUM_BLOCKS = len(gpt2.h)
NUM_HEADS = model.config.n_head
WIDTH = model.config.n_embd
HEAD_SIZE = WIDTH // NUM_HEADS

print("Device:", device)
print("Number of blocks:", NUM_BLOCKS)
print(f"Heads per block: {NUM_HEADS}, numbers per head: {HEAD_SIZE}")

Device: cpu
Number of blocks: 12
Heads per block: 12, numbers per head: 64


## Step 2: Normal run (no changes)

In [2]:
PROMPT = "I live in Paris, so I speak"
OTHER_PROMPT = "I live in Berlin, so I speak"
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)
other_ids = tokenizer(OTHER_PROMPT, return_tensors="pt").input_ids.to(device)
assert other_ids.shape == input_ids.shape
words = [tokenizer.decode(t) for t in input_ids[0]]
print("Word-pieces:", words)

PARIS_POS, LAST_POS = 3, len(words) - 1
FRENCH, GERMAN = tokenizer.encode(" French")[0], tokenizer.encode(" German")[0]

def top_predictions(logits, k=5):
    probs = logits[0, -1].softmax(-1)
    values, indices = probs.topk(k)
    return [(tokenizer.decode(i), v.item()) for i, v in zip(indices, values)]

def chances(logits):
    p = logits[0, -1].softmax(-1)
    return tokenizer.decode(p.argmax()), p[FRENCH].item(), p[GERMAN].item()

def compare(new_logits, title="Modified model"):
    """Print the normal and new top 5 side by side, plus the chance of ' French' and ' German'."""
    print(f"{'Normal model':<26}{title}")
    for (w1, p1), (w2, p2) in zip(top_predictions(normal_logits), top_predictions(new_logits)):
        print(f"  {w1!r:12} {p1:6.2%}      {w2!r:12} {p2:6.2%}")
    _, f0, g0 = chances(normal_logits)
    _, f1, g1 = chances(new_logits)
    print(f"\n  ' French': {f0:6.2%} -> {f1:6.2%}")
    print(f"  ' German': {g0:6.2%} -> {g1:6.2%}")

normal_logits = model(input_ids, use_cache=False).logits
print("\nNormal model - top 5 next words:")
for word, p in top_predictions(normal_logits):
    print(f"  {word!r:12} {p:6.2%}")

Word-pieces: ['I', ' live', ' in', ' Paris', ',', ' so', ' I', ' speak']

Normal model - top 5 next words:
  ' French'    63.05%
  ' English'    4.93%
  ' to'         3.16%
  ' a'          2.94%
  ' with'       1.76%


## Step 3: Split each head into "decide where to look" and "collect"
- `qkv(i, state)` gives each head's query, key and value for every word (64 numbers each).
- `decide(q, k)` gives the **attention weights**: shape (1, 12 heads, word looking, word looked at). Each row adds up to 100%,
  and a word may only look at itself and earlier words.
- `collect(weights, v)` mixes the values using the weights, giving the head outputs.
- `merge_part` joins the 12 heads and adds the result to the table; `think_part` runs the think part (MLP).
- `run_until_weights(ids, layer)` runs blocks before `layer`, then lets the heads of `layer` decide where to look, then **PAUSES**.
  The **checkpoint** holds the table that entered the block, the attention weights, and the values (needed to collect later).
- `resume_after_weights(checkpoint)` **RESUMES**: collects with the (possibly changed) weights, merges, runs the think part, then the remaining blocks.

In [3]:
def embed(input_ids):
    positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
    return gpt2.wte(input_ids) + gpt2.wpe(positions)

def split_heads(t):
    """(1, words, 768) -> (1, 12 heads, words, 64)"""
    return t.view(t.shape[0], t.shape[1], NUM_HEADS, HEAD_SIZE).transpose(1, 2)

def qkv(i, state):
    block = gpt2.h[i]
    q, k, v = block.attn.c_attn(block.ln_1(state)).split(WIDTH, dim=2)
    return split_heads(q), split_heads(k), split_heads(v)

def decide(q, k):
    """Attention weights: how much each word looks at each earlier word, per head. Rows add up to 1."""
    n = q.shape[2]
    scores = q @ k.transpose(-1, -2) / HEAD_SIZE ** 0.5
    later_words = torch.triu(torch.ones(n, n, dtype=torch.bool, device=q.device), diagonal=1)
    return scores.masked_fill(later_words, float("-inf")).softmax(-1)

def collect(weights, v):
    return weights @ v

def merge_part(i, state, head_out):
    merged = head_out.transpose(1, 2).reshape(state.shape)
    return state + gpt2.h[i].attn.c_proj(merged)

def think_part(i, state):
    block = gpt2.h[i]
    return state + block.mlp(block.ln_2(state))

def final_logits(state):
    return model.lm_head(gpt2.ln_f(state))

def run_block(i, state, edit=None):
    q, k, v = qkv(i, state)
    weights = decide(q, k)
    if edit is not None:
        weights = edit(weights)
    return think_part(i, merge_part(i, state, collect(weights, v)))

def run_model(input_ids, edits=None):
    """Full run. `edits` = {block number: function that changes its attention weights}."""
    edits = edits or {}
    state = embed(input_ids)
    for i in range(NUM_BLOCKS):
        state = run_block(i, state, edits.get(i))
    return final_logits(state)

def run_until_weights(input_ids, layer):
    """Run blocks 0..layer-1 fully, then let the heads of block `layer` decide where to look, then PAUSE."""
    state = embed(input_ids)
    for i in range(layer):
        state = run_block(i, state)
    q, k, v = qkv(layer, state)
    return {"layer": layer, "before_look": state, "weights": decide(q, k), "values": v}

def resume_after_weights(checkpoint):
    """RESUME: collect with the (possibly changed) weights, merge, think, then run all later blocks."""
    layer = checkpoint["layer"]
    head_out = collect(checkpoint["weights"], checkpoint["values"])
    state = think_part(layer, merge_part(layer, checkpoint["before_look"], head_out))
    for i in range(layer + 1, NUM_BLOCKS):
        state = run_block(i, state)
    return final_logits(state)

def copy_ckpt(checkpoint, **changes):
    """Separate copy of a checkpoint, optionally with some parts replaced."""
    new = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in checkpoint.items()}
    new.update(changes)
    return new

## Step 4: A tool to change where a head looks
`set_share(weights, heads, pos, target, share)` gives word `target` exactly `share` of the attention of word `pos`, in the chosen heads.
The other earlier words keep their **relative** shares, scaled so everything still adds up to 100%.
- `share=0` **blocks** the head from looking at `target`.
- `share=1` **forces** the head to look only at `target`.

In [4]:
ALL_HEADS = list(range(NUM_HEADS))

def set_share(weights, heads, pos, target, share):
    """Copy of `weights` where, in `heads`, word `pos` gives exactly `share` of its attention to word `target`."""
    weights = weights.clone()
    for h in heads:
        row = weights[0, h, pos]
        rest = row.clone()
        rest[target] = 0
        if rest.sum() > 0:
            rest = rest * (1 - share) / rest.sum()
        rest[target] = share
        weights[0, h, pos] = rest
    return weights

def show_row(weights, h, pos=LAST_POS):
    row = weights[0, h, pos, :pos + 1]
    return "  ".join(f"{w.strip() or w!s}:{s:.0%}" for w, s in zip(words, row))

# quick check of the tool
w = run_until_weights(input_ids, 8)["weights"]
print("Head 8.11, normal   :", show_row(w, 11))
print("Head 8.11, Paris 0% :", show_row(set_share(w, [11], LAST_POS, PARIS_POS, 0), 11))
print("Head 8.11, Paris 100%:", show_row(set_share(w, [11], LAST_POS, PARIS_POS, 1), 11))
assert torch.allclose(set_share(w, [11], LAST_POS, PARIS_POS, 0)[0, 11, LAST_POS].sum(), torch.tensor(1.0))

Head 8.11, normal   : I:40%  live:1%  in:1%  Paris:49%  ,:6%  so:1%  I:0%  speak:1%
Head 8.11, Paris 0% : I:79%  live:2%  in:3%  Paris:0%  ,:11%  so:2%  I:1%  speak:3%
Head 8.11, Paris 100%: I:0%  live:0%  in:0%  Paris:100%  ,:0%  so:0%  I:0%  speak:0%


## Step 5: A counter that records which parts actually run
`heads 5` = the queries, keys and values of block 5 were computed, `merge 5` = its heads were merged, `think 5` = its think part ran.
Deciding and collecting are plain multiplications between those steps.

In [5]:
for h in globals().get("counter_hooks", []):  # safe to rerun this cell
    h.remove()

parts_run = []
counter_hooks = []
for i, block in enumerate(gpt2.h):
    counter_hooks.append(block.attn.c_attn.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"heads {i}")))
    counter_hooks.append(block.attn.c_proj.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"merge {i}")))
    counter_hooks.append(block.mlp.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"think {i}")))
print("Counter attached to", len(counter_hooks), "parts (12 blocks x 3 parts)")

def expected_after(layer):
    return [f"merge {layer}", f"think {layer}"] + [
        f"{p} {i}" for i in range(layer + 1, NUM_BLOCKS) for p in ("heads", "merge", "think")]

Counter attached to 36 parts (12 blocks x 3 parts)


## Step 6: Check that the split run matches the normal model

In [6]:
parts_run.clear()
split_logits = run_model(input_ids)

diff = (split_logits - normal_logits).abs().max().item()
print("Parts run:", len(parts_run), "->", parts_run[:3], "...", parts_run[-3:])
print("Largest difference from normal model:", diff)
assert diff < 1e-3, "Split run does not match the normal model"
print("OK: split run matches the normal model")

Parts run: 36 -> ['heads 0', 'merge 0', 'think 0'] ... ['heads 11', 'merge 11', 'think 11']
Largest difference from normal model: 0.0001068115234375
OK: split run matches the normal model


---
# Part A: head 8.11
In Experiment 5, head 8.11 was the only head in block 8 that looked mainly at ' Paris', and the second most important head overall.

## Step 7: PAUSE after the heads of block 8 decide where to look

In [7]:
PAUSE_A, HEAD_A = 8, 11

parts_run.clear()
ckpt_a = run_until_weights(input_ids, PAUSE_A)  # <-- PAUSE happens here
saved_a = copy_ckpt(ckpt_a)                     # <-- SAVE a separate copy

print(f"PAUSED inside block {PAUSE_A}: heads have decided where to look, nothing collected yet")
print("Parts run so far:", parts_run[-2:], f"... ({len(parts_run)} parts in total)")
print("Attention weights size:", tuple(saved_a["weights"].shape), " (1 sentence, 12 heads, 8 words looking, 8 words looked at)")
print("Values size:           ", tuple(saved_a["values"].shape))
print(f"\nWhere ' speak' looks in head {PAUSE_A}.{HEAD_A}:")
print("  ", show_row(saved_a["weights"], HEAD_A))

PAUSED inside block 8: heads have decided where to look, nothing collected yet
Parts run so far: ['think 7', 'heads 8'] ... (25 parts in total)
Attention weights size: (1, 12, 8, 8)  (1 sentence, 12 heads, 8 words looking, 8 words looked at)
Values size:            (1, 12, 8, 64)

Where ' speak' looks in head 8.11:
   I:40%  live:1%  in:1%  Paris:49%  ,:6%  so:1%  I:0%  speak:1%


## Step 8: RESUME with the saved weights unchanged (check)

In [8]:
parts_run.clear()
resumed = resume_after_weights(copy_ckpt(saved_a))  # <-- RESUME

print("RESUMED. Parts run:", parts_run[:3], "...", parts_run[-3:], f"({len(parts_run)} parts)")
print("Largest difference from normal model:", (resumed - normal_logits).abs().max().item())
assert parts_run == expected_after(PAUSE_A), "Something earlier was rerun!"
assert torch.allclose(resumed, normal_logits, atol=1e-3)
print("OK: resuming gives the normal answer, and block 8's heads were not rerun")

RESUMED. Parts run: ['merge 8', 'think 8', 'heads 9'] ... ['heads 11', 'merge 11', 'think 11'] (11 parts)
Largest difference from normal model: 0.0001068115234375
OK: resuming gives the normal answer, and block 8's heads were not rerun


## Step 9: MODIFY — turn the attention of head 8.11 on ' Paris' up and down
We set the share ' speak' gives to ' Paris' in head 8.11 to different values, from 0% (blocked) to 100% (forced).
Each line is a separate resume from the same saved checkpoint.

In [9]:
def resume_with_weights(saved, weights):
    parts_run.clear()
    logits = resume_after_weights(copy_ckpt(saved, weights=weights))  # <-- RESUME
    assert parts_run == expected_after(saved["layer"])
    return logits

def share_sweep(saved, heads, shares=(0, 0.1, 0.25, None, 0.5, 0.75, 1.0)):
    normal_share = saved["weights"][0, heads[0], LAST_POS, PARIS_POS].item()
    print(f"{'Share on Paris':<18}{'top word':<12}{'French':>8}{'German':>8}")
    for share in shares:
        label = f"{normal_share:.0%} (normal)" if share is None else f"{share:.0%}"
        share = normal_share if share is None else share
        logits = resume_with_weights(saved, set_share(saved["weights"], heads, LAST_POS, PARIS_POS, share))  # <-- MODIFY
        top, f, g = chances(logits)
        print(f"{label:<18}{top!r:<12}{f:>8.1%}{g:>8.1%}")

share_sweep(saved_a, [HEAD_A])

Share on Paris    top word      French  German
0%                ' French'      51.0%    0.6%
10%               ' French'      53.3%    0.6%
25%               ' French'      57.3%    0.6%
49% (normal)      ' French'      63.0%    0.6%
50%               ' French'      63.2%    0.6%
75%               ' French'      65.7%    0.6%
100%              ' French'      64.8%    0.6%


## Step 10: MODIFY — redirect head 8.11 to look only at another word
Instead of ' Paris', head 8.11 of ' speak' now looks 100% at one other word.

In [10]:
print(f"{'Looks only at':<16}{'top word':<12}{'French':>8}{'German':>8}")
for target in range(LAST_POS + 1):
    logits = resume_with_weights(saved_a, set_share(saved_a["weights"], [HEAD_A], LAST_POS, target, 1.0))
    top, f, g = chances(logits)
    print(f"{words[target]!r:<16}{top!r:<12}{f:>8.1%}{g:>8.1%}")

Looks only at   top word      French  German
'I'             ' French'      48.9%    0.6%
' live'         ' French'      62.3%    0.3%
' in'           ' French'      63.1%    0.6%
' Paris'        ' French'      64.8%    0.6%
','             ' French'      59.1%    0.7%
' so'           ' French'      43.3%    0.8%
' I'            ' French'      47.4%    0.7%
' speak'        ' French'      45.0%    0.5%


---
# Part B: head 11.2, and both heads together
Head 11.2 was the most important head in Experiment 5. Swapping only the keys in Experiment 4 raised its attention on ' Paris' from 28% to 54%
and raised ' French' to 83%. Here we set that share directly.

## Step 11: PAUSE in block 11 and turn the attention of head 11.2 on ' Paris' up and down

In [11]:
PAUSE_B, HEAD_B = 11, 2

ckpt_b = run_until_weights(input_ids, PAUSE_B)  # <-- PAUSE
saved_b = copy_ckpt(ckpt_b)                     # <-- SAVE
print(f"Where ' speak' looks in head {PAUSE_B}.{HEAD_B}:")
print("  ", show_row(saved_b["weights"], HEAD_B))
print()
share_sweep(saved_b, [HEAD_B], shares=(0, 0.1, None, 0.4, 0.54, 0.75, 1.0))

Where ' speak' looks in head 11.2:
   I:64%  live:1%  in:2%  Paris:28%  ,:3%  so:2%  I:1%  speak:0%

Share on Paris    top word      French  German
0%                ' French'      35.4%    1.4%
10%               ' French'      45.4%    1.1%
28% (normal)      ' French'      63.0%    0.6%
40%               ' French'      72.1%    0.4%
54%               ' French'      80.4%    0.2%
75%               ' French'      88.2%    0.1%
100%              ' French'      92.8%    0.0%


## Step 12: Block both heads from looking at ' Paris', in the same run
We pause in block 8 **and** in block 11 in one run, and set the share on ' Paris' to 0% for head 8.11 and head 11.2.
For comparison, we also block **all 12 heads** of blocks 8 and 11, and all heads of all blocks.

In [12]:
def block_paris(heads, positions=(LAST_POS,)):
    """Edit for run_model: in the given heads, the given words give 0% of their attention to ' Paris'."""
    def edit(weights):
        for pos in positions:
            weights = set_share(weights, heads, pos, PARIS_POS, 0)
        return weights
    return edit

trials = [
    ("head 8.11", {8: block_paris([11])}),
    ("head 11.2", {11: block_paris([2])}),
    ("heads 8.11 + 11.2", {8: block_paris([11]), 11: block_paris([2])}),
    ("all heads of blocks 8, 11", {8: block_paris(ALL_HEADS), 11: block_paris(ALL_HEADS)}),
    ("all heads, all blocks", {i: block_paris(ALL_HEADS) for i in range(NUM_BLOCKS)}),
]
print(f"{'Blocked from Paris (row of speak)':<36}{'top word':<12}{'French':>8}{'German':>8}")
for name, edits in trials:
    parts_run.clear()
    logits = run_model(input_ids, edits=edits)
    assert len(parts_run) == 3 * NUM_BLOCKS, "Each part should run exactly once"
    top, f, g = chances(logits)
    print(f"{name:<36}{top!r:<12}{f:>8.1%}{g:>8.1%}")

Blocked from Paris (row of speak)   top word      French  German
head 8.11                           ' French'      51.0%    0.6%
head 11.2                           ' French'      35.4%    1.4%
heads 8.11 + 11.2                   ' French'      26.8%    1.2%
all heads of blocks 8, 11           ' French'      43.6%    0.9%
all heads, all blocks               ' English'      9.6%    2.4%


---
# Part C: cutting the link to ' Paris'

## Step 13: Cut the link from ' speak' to ' Paris' in one block at a time
In each block separately, all 12 heads of ' speak' get 0% on ' Paris'.
In the second column, **every** word after ' Paris' (', so I speak') is cut off from it in that block.

In [13]:
after_paris = tuple(range(PARIS_POS + 1, LAST_POS + 1))
print(f"{'Block':<7}{'speak only':>18}{'all later words':>20}")
print(f"{'':<7}{'French/German':>18}{'French/German':>20}")
for layer in range(NUM_BLOCKS):
    ckpt = run_until_weights(input_ids, layer)
    line = f"{layer:<7}"
    for positions in ((LAST_POS,), after_paris):
        weights = ckpt["weights"]
        for pos in positions:
            weights = set_share(weights, ALL_HEADS, pos, PARIS_POS, 0)
        _, f, g = chances(resume_with_weights(ckpt, weights))
        line += f"{f:>12.1%}/{g:<5.1%}"
    print(line)

Block          speak only     all later words
            French/German       French/German
0             63.5%/0.6%        63.3%/0.6% 
1             63.6%/0.6%        63.5%/0.6% 
2             61.8%/0.6%        62.0%/0.6% 
3             62.9%/0.6%        62.8%/0.6% 
4             62.9%/0.6%        63.3%/0.6% 
5             62.9%/0.6%        63.5%/0.6% 
6             63.7%/0.6%        64.6%/0.6% 
7             62.8%/0.6%        62.7%/0.6% 
8             51.2%/0.5%        51.5%/0.5% 
9             68.5%/0.7%        68.5%/0.7% 
10            76.6%/0.4%        76.1%/0.4% 
11            49.2%/1.2%        49.2%/1.2% 


## Step 14: Cut every link to ' Paris', in every block
Now no word after ' Paris' can look at it, in any head of any block. The word ' Paris' is still in the sentence,
but nothing can read it, so the model should behave as if it was never there.
For comparison, we run the sentence with ' Paris' replaced by other cities.

In [14]:
parts_run.clear()
logits_cut = run_model(input_ids, edits={i: block_paris(ALL_HEADS, after_paris) for i in range(NUM_BLOCKS)})
assert len(parts_run) == 3 * NUM_BLOCKS
compare(logits_cut, "Nothing can read ' Paris'")

print(f"\n{'Same sentence with':<22}{'top word':<12}{'French':>8}{'German':>8}")
for city in [" Paris", " Berlin", " London", " Madrid", " Tokyo"]:
    ids = tokenizer(f"I live in{city}, so I speak", return_tensors="pt").input_ids.to(device)
    top, f, g = chances(model(ids, use_cache=False).logits)
    print(f"{city!r:<22}{top!r:<12}{f:>8.1%}{g:>8.1%}")
top, f, g = chances(logits_cut)
print(f"{'(Paris unreadable)':<22}{top!r:<12}{f:>8.1%}{g:>8.1%}")

Normal model              Nothing can read ' Paris'
  ' French'    63.05%      ' English'   15.64%
  ' English'    4.93%      ' to'        10.92%
  ' to'         3.16%      ' for'        7.58%
  ' a'          2.94%      ' Spanish'    5.59%
  ' with'       1.76%      ' the'        3.61%

  ' French': 63.05% ->  2.07%
  ' German':  0.62% ->  1.07%

Same sentence with    top word      French  German
' Paris'              ' French'      63.0%    0.6%
' Berlin'             ' German'       0.8%   63.6%
' London'             ' English'      1.8%    1.1%
' Madrid'             ' Spanish'      0.7%    0.4%
' Tokyo'              ' Japanese'     0.3%    0.3%
(Paris unreadable)    ' English'      2.1%    1.1%
